In [4]:
from dotenv import load_dotenv

In [5]:
load_dotenv("../.env")

True

In [6]:
API_KEY = os.getenv("OPENAI_API_KEY")

In [7]:
import os
import re
import uuid
import pickle
from typing import List, Dict, Any, Sequence

from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import Callbacks
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [8]:
SOURCE_DOCS = [
    "../info-mascotas/guia-cuidado.md", 
    "../info-mascotas/Tenencia-Responsable.md"
]
CHROMA_PATH = "./chroma_db"
DOCSTORE_PATH = "docstore_parents.pkl"
BM25_PATH = "bm25_index.pkl"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

In [9]:
def get_openai_key():
    key = os.getenv("OPENAI_API_KEY")
    if not key: raise ValueError("❌ Falta OPENAI_API_KEY")
    return key

In [10]:
def clean_content(text: str) -> str:
    text = re.sub(r'^## Page \d+\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'^---\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'(\w+)[-\u2010-\u2015]\s*\n\s*(\w+)', r'\1\2', text)
    text = re.sub(r'(\w+)-\s*\n\s*(\|)', r'\1\n\2', text)
    text = text.replace("– Partner In Paw", "").replace("I****ncluye", "Incluye")
    text = re.sub(r'con la siguiente imagen te mostramos.*?:', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\* \* \*', '', text)
    text = re.sub(r'^\s*[\*]{2,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*\*(.+?)\*\*\n-+', r'# \1', text)
    text = re.sub(r'^\s*\*\*(.+?)\*\*[:]?\s*$', r'## \1', text, flags=re.MULTILINE)
    text = re.sub(r'^(?!#)(Módulo [IVX]+.*)', r'# \1', text, flags=re.MULTILINE)
    text = re.sub(r'^(?!#)([IVX]+\.-\s.*)', r'## \1', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

In [11]:
class ScoreInjectedReranker(CrossEncoderReranker):
    def compress_documents(
        self,
        documents: Sequence[Document],
        query: str,
        callbacks: Callbacks = None,
    ) -> Sequence[Document]:
        if len(documents) == 0:
            return []
        pairs = [(query, doc.page_content) for doc in documents]
        scores = self.model.score(pairs)
        docs_with_scores = list(zip(documents, scores))
        result = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)
        final_results = []
        for doc, score in result[:self.top_n]:
            new_doc = Document(
                page_content=doc.page_content,
                metadata=doc.metadata.copy()
            )
            new_doc.metadata["relevance_score"] = float(score)
            final_results.append(new_doc)         
        return final_results

In [12]:
class CustomHybridParentRetriever(BaseRetriever):
    child_retriever: Any
    docstore: Dict[str, Document] = {}
    id_key: str = "parent_id"

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        optimized_children = self.child_retriever.invoke(query)
        
        parent_map = {}
        
        for child in optimized_children:
            p_id = child.metadata.get(self.id_key)
            score = child.metadata.get("relevance_score", -999.0)
            
            if p_id:
                if p_id not in parent_map:
                    parent_map[p_id] = score
                else:
                    parent_map[p_id] = max(parent_map[p_id], score)

        final_docs = []
        for p_id, score in parent_map.items():
            if p_id in self.docstore:
                parent_doc = self.docstore[p_id]
                
                enriched_doc = Document(
                    page_content=parent_doc.page_content,
                    metadata=parent_doc.metadata.copy()
                )
                
                enriched_doc.metadata["retrieved_parent_id"] = p_id
                enriched_doc.metadata["final_score"] = score
                
                header_context = ""
                meta = enriched_doc.metadata
                if "Module" in meta: header_context += f"# {meta['Module']}\n"
                if "Section" in meta: header_context += f"## {meta['Section']}\n"
                if "Topic" in meta: header_context += f"# {meta['Topic']}\n"
                if "Subtopic" in meta: header_context += f"## {meta['Subtopic']}\n"
                
                if header_context:
                    enriched_doc.page_content = header_context + "\n" + enriched_doc.page_content

                final_docs.append(enriched_doc)
        
        final_docs.sort(key=lambda x: x.metadata.get("final_score", -999), reverse=True)
        
        return final_docs

In [13]:
def get_vectorstore_connection():
    return Chroma(
        collection_name="rag_final",
        embedding_function=OpenAIEmbeddings(api_key=get_openai_key()),
        persist_directory=CHROMA_PATH
    )

In [14]:
def run_ingest():
    print("🚀 MODO INGESTA: Procesando documentos...")
    # Configurar Splitters
    parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
    child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
    md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Title"), ("##", "Subtitle")])
    global_docstore = {}
    global_child_docs = []
    for file_name in SOURCE_DOCS:
        if not os.path.exists(file_name):
            print(f"⚠️  Saltando {file_name} (No existe)")
            continue
        print(f"   📂 Leyendo {file_name}...")
        with open(file_name, "r", encoding="utf-8") as f: raw_text = f.read()
        clean_text = clean_content(raw_text)
        base_docs = md_splitter.split_text(clean_text)
        parent_docs = parent_splitter.split_documents(base_docs)
        for parent in parent_docs:
            p_id = str(uuid.uuid4())
            parent.metadata["source"] = file_name
            global_docstore[p_id] = parent # Guardar Padre 
            children = child_splitter.split_documents([parent])
            for child in children:
                child.metadata["parent_id"] = p_id
                child.metadata["source"] = file_name
                global_child_docs.append(child)
    if not global_docstore:
        print("❌ Error: No se procesaron documentos.")
        return
    print("💾 Guardando en disco...")
    # 1. Guardar Padres (Pickle)
    with open(DOCSTORE_PATH, "wb") as f: pickle.dump(global_docstore, f)
    # 2. Guardar BM25 (Hijos)
    print("   🧮 Calculando BM25...")
    bm25_retriever = BM25Retriever.from_documents(global_child_docs)
    bm25_retriever.k = 20 # Pedimos 20 candidatos iniciales para el Reranker
    with open(BM25_PATH, "wb") as f: pickle.dump(bm25_retriever, f)
    # 3. Guardar Vectores (Chroma)
    print("   🧠 Generando Embeddings (Chroma)...")
    vector_db = Chroma(
        collection_name="rag_final",
        embedding_function=OpenAIEmbeddings(api_key=get_openai_key()),
        persist_directory=CHROMA_PATH
    )
    vector_db.add_documents(global_child_docs)
    print("✨ Ingesta completada.")

In [ ]:
def run_query_loop():
    print("\n🔄 CARGANDO SISTEMA...")
    if not os.path.exists(DOCSTORE_PATH):
        print("❌ Ejecuta ingesta primero.")
        return

    with open(DOCSTORE_PATH, "rb") as f: docstore = pickle.load(f)
    with open(BM25_PATH, "rb") as f: bm25_retriever = pickle.load(f)
    
    vector_db = get_vectorstore_connection()
    chroma_retriever = vector_db.as_retriever(search_kwargs={"k": 20})

    # Ensamble
    ensemble_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, chroma_retriever], weights=[0.4, 0.6]
    )

    print("   ⚖️  Cargando Reranker...")
    model = HuggingFaceCrossEncoder(model_name=RERANKER_MODEL)
    
    compressor = ScoreInjectedReranker(model=model, top_n=3)
    
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=ensemble_retriever
    )

    final_retriever = CustomHybridParentRetriever(child_retriever=compression_retriever)
    final_retriever.docstore = docstore

    print("✅ LISTO.\n")

    while True:
        query = input("user> ")
        if query.lower() in ["salir", "exit"]: break
        
        docs = final_retriever.invoke(query)
        if not docs: print("   🤖 Nada relevante."); continue

        print(f"\n✅ Resultados ({len(docs)} padres):\n")
        for i, doc in enumerate(docs):
            meta = doc.metadata
            print(f"--- [Doc {i+1}] ID: {meta.get('retrieved_parent_id')} ---")
            # AHORA SÍ DEBERÍA VERSE EL NÚMERO
            print(f"Score Rerank: {meta.get('final_score'):.4f}") 
            print(f"Fuente: {meta.get('source')}")
            print("-" * 20)
            print(doc.page_content + "...\n")
            print("="*50)

if __name__ == "__main__":
    if not os.path.exists(DOCSTORE_PATH):
        run_ingest()
        pass
    run_query_loop()


🔄 CARGANDO SISTEMA...
   ⚖️  Cargando Reranker...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ LISTO.



user>  niveles de estres en perros



✅ Resultados (3 padres):

--- [Doc 1] ID: 9e394f02-3662-4e47-8d62-7672a27ca9d6 ---
Score Rerank: 0.6072
Fuente: guia-cuidado.md
--------------------
# Remedios Naturales, Técnicas y Tips para tranquilizar a tu Perro y Gat

La ansiedad y el estrés en nuestras mascotas pueden ser un desafío tanto para ellas como para nosotros. Reconocer las señales de estrés y ansiedad en tu mascota es esencial para poder actuar a tiempo y aliviar su malestar. Aunque perros y gatos pueden expresar su ansiedad de diferentes maneras, hay comportamientos comunes que indican que algo no está bien....

--- [Doc 2] ID: 84128b2a-029e-4b60-999c-6cf199ae7d27 ---
Score Rerank: 0.3504
Fuente: guia-cuidado.md
--------------------
# Señales de Estrés, Ansiedad y Miedo
## En Perros:

*   **Ladridos o aullidos excesivos:** Los perros pueden vocalizar más de lo normal cuando están ansiosos o estresados.
*   **Jadeo excesivo:** Si tu perro jadea en ausencia de calor o ejercicio, podría estar ansioso.
*   **Temblores o s